# 🔍 Exploratory Data Analysis — UNSW-NB15 Dataset

This notebook investigates the UNSW-NB15 training dataset to understand:
- Data shape, types, and missing values
- Class distribution and imbalance
- Attack category breakdown
- Feature correlations
- Key discriminative features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully ✅')

## 1. Load Dataset

In [ ]:
# Load data - prefer parquet if available
DATA_DIR = 'data'
parquet_path = os.path.join(DATA_DIR, 'UNSW_NB15_training-set.parquet')
csv_path = os.path.join(DATA_DIR, 'UNSW_NB15_training-set.csv')

if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)
    print(f'Loaded from Parquet ✅')
else:
    df = pd.read_csv(csv_path)
    # Clean attack_cat whitespace
    df['attack_cat'] = df['attack_cat'].astype(str).str.strip()
    print(f'Loaded from CSV ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 2. Data Overview

In [ ]:
# Data types
print('=== Data Types ===')
print(df.dtypes.value_counts())
print(f'\nTotal columns: {len(df.columns)}')
print(f'\nColumn names:\n{list(df.columns)}')

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_info = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
missing_info = missing_info[missing_info['Missing'] > 0].sort_values('Missing', ascending=False)

if len(missing_info) > 0:
    print('=== Missing Values ===')
    print(missing_info)
else:
    print('No missing values found ✅')

# Duplicates
dup_count = df.duplicated().sum()
print(f'\nDuplicate rows: {dup_count:,} ({dup_count/len(df)*100:.2f}%)')

In [ ]:
# Statistical summary of numerical features
df.describe().T.round(2)

## 3. Class Distribution (Binary: label)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
label_counts = df['label'].value_counts()
colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(['Normal (0)', 'Attack (1)'], label_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Binary Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12)
for bar, count in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{count:,}', ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(label_counts.values, labels=['Normal', 'Attack'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=(0.02, 0.02), shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Imbalance ratio
ratio = label_counts[0] / label_counts[1]
print(f'Normal: {label_counts[0]:,}  |  Attack: {label_counts[1]:,}')
print(f'Imbalance ratio (Normal:Attack): {ratio:.2f}:1')

## 4. Attack Category Breakdown (Multi-class: attack_cat)

In [ ]:
attack_counts = df['attack_cat'].value_counts()

fig, ax = plt.subplots(figsize=(14, 7))
palette = sns.color_palette('viridis', len(attack_counts))
bars = ax.barh(attack_counts.index[::-1], attack_counts.values[::-1], color=palette[::-1], edgecolor='white')
ax.set_xlabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Attack Category Distribution (Training Set)', fontsize=14, fontweight='bold', pad=15)

for bar, count in zip(bars, attack_counts.values[::-1]):
    pct = count / len(df) * 100
    ax.text(bar.get_width() + max(attack_counts) * 0.01,
            bar.get_y() + bar.get_height()/2,
            f'{count:,} ({pct:.1f}%)', va='center', fontsize=10)

plt.tight_layout()
plt.show()

# Print table
print('\n=== Attack Category Counts ===')
for cat, count in attack_counts.items():
    print(f'  {cat:20s}: {count:6,} ({count/len(df)*100:5.1f}%)')

## 5. Categorical Feature Analysis

In [ ]:
cat_cols = ['proto', 'service', 'state']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, col in enumerate(cat_cols):
    top_vals = df[col].value_counts().head(10)
    sns.barplot(x=top_vals.values, y=top_vals.index, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Top 10: {col}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Count')
    print(f'{col}: {df[col].nunique()} unique values')

plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
# Select numerical columns only
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[num_cols].corr()

# Find highly correlated pairs (|r| > 0.9)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

print(f'Highly correlated feature pairs (|r| > 0.9): {len(high_corr_pairs)}')
for f1, f2, r in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
    print(f'  {f1:20s} ↔ {f2:20s}  r = {r:.4f}')

In [ ]:
# Heatmap of correlation with target
fig, ax = plt.subplots(figsize=(16, 12))

# Use mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.7})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Feature Correlation with Target

In [ ]:
# Correlation of each feature with the binary label
target_corr = corr_matrix['label'].drop('label').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(target_corr)))
bars = ax.barh(target_corr.index[::-1], target_corr.values[::-1], color=colors[::-1])
ax.set_xlabel('|Correlation with Label|', fontsize=12, fontweight='bold')
ax.set_title('Feature Importance by Correlation with Attack Label', fontsize=14, fontweight='bold', pad=15)
ax.axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='r=0.3 threshold')
ax.legend()
plt.tight_layout()
plt.show()

print('\nTop 10 features most correlated with attack label:')
for feat, corr in target_corr.head(10).items():
    print(f'  {feat:20s}: |r| = {corr:.4f}')

## 8. Feature Distributions by Class

In [ ]:
# Top 6 discriminative features
top_features = target_corr.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    for label_val, color, name in [(0, '#2ecc71', 'Normal'), (1, '#e74c3c', 'Attack')]:
        subset = df[df['label'] == label_val][feat]
        # Clip extreme outliers for visualization
        q99 = subset.quantile(0.99)
        subset = subset[subset <= q99]
        axes[i].hist(subset, bins=50, alpha=0.5, color=color, label=name, density=True)
    axes[i].set_title(f'{feat}', fontsize=12, fontweight='bold')
    axes[i].legend()
    axes[i].set_ylabel('Density')

plt.suptitle('Feature Distributions: Normal vs Attack', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Key Findings Summary

### Class Imbalance
- The dataset has a moderate class imbalance between Normal and Attack traffic
- Among attack categories, **Generic**, **Exploits**, and **Fuzzers** are the most common
- **Worms**, **Shellcode**, and **Backdoor** are rare categories — models may struggle with these

### Highly Correlated Features
- Several feature pairs have |r| > 0.9, indicating redundancy
- Notable pairs: `sload`↔`sloss`, `ct_srv_src`↔`ct_dst_ltm`, etc.
- These don't need to be removed for tree-based models (Decision Tree, XGBoost) but may affect Logistic Regression

### Discriminative Features
- `sttl`, `ct_state_ttl`, `dttl`, `ct_srv_src` show the strongest correlation with the attack label
- These features will likely be the most important for classification

### Categorical Features
- `proto`: Dominated by TCP/UDP but includes many rare protocols
- `service`: Highly skewed — most records have `-` (no service)
- `state`: Several connection states; some are strong attack indicators